# Consolidated Dashcam Pipeline

This notebook processes one dashcam video and produces:

- One annotated video where each detected vehicle has exactly one output class: `gas_car`, `gas_bus`, `gas_truck`, or `ev`.
- One CSV with separate interval count columns for every class/proximity/status combination.

It integrates the functionality from:

- `Code/dashcam_vehicle_proximity_analysis.ipynb`
- `Code/EV experiement/EV_detection_from_dashcam_3.0.ipynb`

This notebook runs a single Stage 1 YOLO + ByteTrack pass to localize `car`, `bus`, and `truck`, then runs the local fine-tuned EV model on tracked vehicle crops. Confirmed EV tracks are labeled and counted only as `ev`; otherwise Stage 1 vehicles are labeled as `gas_car`, `gas_bus`, or `gas_truck`.

## 1) Environment and Imports

Use the same environment as the EV notebook, preferably the `cv` conda environment. Uncomment the install cell only if a package is missing.

In [ ]:
# Uncomment only if running in a fresh environment:
# !pip install ultralytics opencv-python pandas numpy tqdm inference roboflow

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path

import cv2  # pyright: ignore[reportMissingImports]
import numpy as np  # pyright: ignore[reportMissingImports]
import pandas as pd  # pyright: ignore[reportMissingImports]
from tqdm.auto import tqdm  # pyright: ignore[reportMissingModuleSource]
from ultralytics import YOLO  # pyright: ignore[reportMissingImports]

## 2) Configuration

Set `VIDEO_PATH` and `START_DATETIME` before running.

In [ ]:
WORKSPACE_ROOT = Path('..').resolve()  # repo root when notebook runs from Code/

# ===== Required input =====
VIDEO_PATH = WORKSPACE_ROOT / 'Artifacts/input/dashcam_sample.mp4'
START_DATETIME = '2026-01-01 00:00:00+00:00'

# ===== Output files =====
CONSOLIDATED_DIR = WORKSPACE_ROOT / 'Artifacts/Consolidated_dashcam_pipeline'
PROXIMITY_COUNTS_DIR = CONSOLIDATED_DIR / 'Proximity_counts'
ANNOTATED_VIDEOS_DIR = CONSOLIDATED_DIR / 'Annotated_videos'
PROXIMITY_COUNTS_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV_PATH = PROXIMITY_COUNTS_DIR / 'dashcam_sample_consolidated_proximity_counts.csv'
OUTPUT_VIDEO_PATH = ANNOTATED_VIDEOS_DIR / 'dashcam_sample_consolidated_annotated.mp4'

# ===== Sampling, proximity, and stopped-vehicle controls =====
SAMPLE_EVERY_SECONDS = 30
CLOSE_AREA_THRESHOLD_PX2 = 30000
MEDIUM_AREA_THRESHOLD_PX2 = 14000
STOPPED_SECONDS = 4.0
STATIONARY_CENTER_DELTA_PX = 12.0
STATIONARY_AREA_CHANGE_RATIO = 0.08
SCENE_STATIC_MAX_MOTION_PX = 0.8

# ===== Stage 1: YOLO vehicle detection + tracking =====
MODEL_WEIGHTS = str(WORKSPACE_ROOT / 'Code/yolov8n.pt')
CONFIDENCE_THRESHOLD = 0.70
TRACKER_CONFIG = 'bytetrack.yaml'
VEHICLE_CLASSES = ('car', 'bus', 'truck')
GAS_CLASS_NAMES = {'car': 'gas_car', 'bus': 'gas_bus', 'truck': 'gas_truck'}

# ===== Stage 2: local fine-tuned EV YOLO detector =====
EV_MODEL_DIR = WORKSPACE_ROOT / 'Code/EV_experiment/models/ev-car-vixid'
EV_MODEL_WEIGHTS = EV_MODEL_DIR / 'weights/best.pt'
EV_CONFIDENCE_THRESHOLD = 0.70
EV_INFERENCE_CONFIDENCE = 0.25  # internal detector threshold; lower finds more candidate boxes
EV_CROP_PAD_RATIO = 0.35  # the EV detector benefits from context around the vehicle box
EV_CLASSIFY_EVERY_N_FRAMES = 5
EV_MIN_VOTES = 5
EV_MIN_CROP_SIDE_PX = 64
NON_EV_CLASS_NAMES = {'Non_EV', 'Non-EV', 'NON-EV', 'MG (Non-EV)'}
EV_CLASS_NAME = 'ev'

# Optional one-time download settings. Prefer setting ROBOFLOW_API_KEY in the environment.
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
ROBOFLOW_WORKSPACE = 'your-roboflow-workspace'
ROBOFLOW_PROJECT = 'ev-car-vixid'
ROBOFLOW_VERSION = 1

OUTPUT_CLASSES = (*GAS_CLASS_NAMES.values(), EV_CLASS_NAME)
PROXIMITY_LEVELS = ('close', 'medium', 'far')
PROXIMITY_PRIORITY = {'far': 0, 'medium': 1, 'close': 2}
PROXIMITY_COLORS = {
    'close': (0, 255, 0),
    'medium': (0, 255, 255),
    'far': (0, 165, 255),
}
EV_OVERLAY_COLOR = (255, 0, 255)


## 3) Shared Helpers

These helpers preserve the original proximity thresholds, stopped-vehicle logic inputs, interval aggregation behavior, and CSV row format while using mutually exclusive output classes: `gas_car`, `gas_bus`, `gas_truck`, and `ev`.

In [ ]:
def classify_proximity(bbox_area: float, close_threshold: float, medium_threshold: float) -> str:
    """Return close/medium/far from bbox area thresholds."""
    if close_threshold < medium_threshold:
        raise ValueError('CLOSE_AREA_THRESHOLD_PX2 must be >= MEDIUM_AREA_THRESHOLD_PX2')

    if bbox_area >= close_threshold:
        return 'close'
    if bbox_area >= medium_threshold:
        return 'medium'
    return 'far'


def csv_count_columns() -> list[str]:
    """Return ordered count columns for every class/proximity/status combination."""
    columns: list[str] = []
    for class_name in OUTPUT_CLASSES:
        for proximity in PROXIMITY_LEVELS:
            columns.append(f'{class_name}_{proximity}')
            columns.append(f'{class_name}_{proximity}_stopped')
    return columns


CSV_COLUMNS = ['timestamp', *csv_count_columns()]


def init_interval_counter() -> dict[str, int]:
    """Initialize count dictionary for one interval."""
    return {column: 0 for column in csv_count_columns()}


def build_interval_counts_from_tracks(unique_tracks: dict) -> dict[str, int]:
    """Build one interval count object from unique track records.

    Base columns count moving vehicles; *_stopped columns count stopped vehicles.
    """
    counts = init_interval_counter()

    for record in unique_tracks.values():
        class_name = record['class_name']
        proximity = record['proximity']
        stopped = record['stopped']

        base_key = f'{class_name}_{proximity}'
        stopped_key = f'{base_key}_stopped'
        output_key = stopped_key if stopped else base_key
        if output_key in counts:
            counts[output_key] += 1

    return counts


def interval_row_template(timestamp: str, counts: dict[str, int]) -> dict[str, int | str]:
    """Create a standardized row for CSV output."""
    return {'timestamp': timestamp, **{column: counts[column] for column in csv_count_columns()}}


def estimate_scene_motion_px(prev_img_gray: np.ndarray | None, curr_img_gray: np.ndarray | None) -> float:
    """Estimate global scene motion in pixels using sparse optical flow."""
    if prev_img_gray is None or curr_img_gray is None:
        return float('inf')

    points_prev = cv2.goodFeaturesToTrack(
        prev_img_gray,
        maxCorners=260,
        qualityLevel=0.01,
        minDistance=8,
        blockSize=7,
    )
    if points_prev is None or len(points_prev) < 20:
        return float('inf')

    points_curr, status, _ = cv2.calcOpticalFlowPyrLK(prev_img_gray, curr_img_gray, points_prev, None)
    if points_curr is None or status is None:
        return float('inf')

    valid_prev = points_prev[status.flatten() == 1]
    valid_curr = points_curr[status.flatten() == 1]
    if len(valid_prev) < 20:
        return float('inf')

    displacement = np.linalg.norm(valid_curr - valid_prev, axis=1)
    return float(np.median(displacement))


def format_wall_clock(ts: datetime) -> str:
    """Format as ISO 8601 UTC with offset, matching the source notebooks."""
    if ts.tzinfo is None:
        ts = ts.replace(tzinfo=timezone.utc)
    else:
        ts = ts.astimezone(timezone.utc)
    return ts.isoformat(sep=' ', timespec='seconds')


def update_motion_state(
    track_motion_state: dict[tuple[str, int], dict],
    track_key: tuple[str, int],
    *,
    center: tuple[float, float],
    bbox_area: float,
    timestamp_float: float,
    scene_is_static: bool,
) -> dict:
    """Update stopped/moving state for one tracked class instance."""
    cx, cy = center
    state = track_motion_state.get(track_key)
    if state is None:
        state = {
            'last_center': (cx, cy),
            'last_area': float(bbox_area),
            'last_timestamp': timestamp_float,
            'stationary_time': 0.0,
            'stopped': False,
        }
    else:
        prev_cx, prev_cy = state['last_center']
        prev_area = max(1.0, state['last_area'])
        dt = max(0.0, timestamp_float - state['last_timestamp'])

        center_displacement = float(np.hypot(cx - prev_cx, cy - prev_cy))
        area_change_ratio = abs(float(bbox_area) - prev_area) / prev_area
        vehicle_box_is_stable = (
            center_displacement <= STATIONARY_CENTER_DELTA_PX
            and area_change_ratio <= STATIONARY_AREA_CHANGE_RATIO
        )

        if vehicle_box_is_stable and scene_is_static:
            state['stationary_time'] += dt
        else:
            state['stationary_time'] = 0.0
            state['stopped'] = False

        if state['stationary_time'] >= STOPPED_SECONDS:
            state['stopped'] = True

        state['last_center'] = (cx, cy)
        state['last_area'] = float(bbox_area)
        state['last_timestamp'] = timestamp_float

    track_motion_state[track_key] = state
    return state


def update_interval_record(
    interval_records: dict[tuple[str, int], dict],
    track_key: tuple[str, int],
    *,
    class_name: str,
    proximity: str,
    stopped: bool,
) -> None:
    """Count each track once per interval, keeping nearest proximity and any stopped evidence."""
    if track_key not in interval_records:
        interval_records[track_key] = {
            'class_name': class_name,
            'proximity': proximity,
            'stopped': stopped,
        }
        return

    existing = interval_records[track_key]['proximity']
    if PROXIMITY_PRIORITY[proximity] > PROXIMITY_PRIORITY[existing]:
        interval_records[track_key]['proximity'] = proximity
    interval_records[track_key]['stopped'] = interval_records[track_key]['stopped'] or stopped


def promote_track_records_to_ev(
    interval_unique_tracks: dict[int, dict[tuple[str, int], dict]],
    *,
    gas_class_name: str,
    track_id: int,
) -> None:
    """Replace gas records for a confirmed EV track across stored intervals."""
    gas_key = (gas_class_name, track_id)
    ev_key = (EV_CLASS_NAME, track_id)

    for interval_records in interval_unique_tracks.values():
        gas_record = interval_records.pop(gas_key, None)
        if gas_record is None:
            continue

        if ev_key not in interval_records:
            interval_records[ev_key] = {
                'class_name': EV_CLASS_NAME,
                'proximity': gas_record['proximity'],
                'stopped': gas_record['stopped'],
            }
            continue

        existing = interval_records[ev_key]['proximity']
        if PROXIMITY_PRIORITY[gas_record['proximity']] > PROXIMITY_PRIORITY[existing]:
            interval_records[ev_key]['proximity'] = gas_record['proximity']
        interval_records[ev_key]['stopped'] = interval_records[ev_key]['stopped'] or gas_record['stopped']


## 4) EV Model Helpers

This section preserves the local fine-tuned EV detector flow from the EV notebook: resolve/download weights, crop tracked vehicles with padding, run the local YOLO EV model, and confirm EV tracks with voting plus an EMA score.

In [ ]:
def find_ev_weights(search_root: Path) -> Path | None:
    """Return the best available EV model checkpoint under search_root."""
    if not search_root.exists():
        return None

    for name in ('best.pt', 'weights.pt', 'weights.onnx'):
        matches = sorted(search_root.rglob(name))
        if matches:
            return matches[0]

    pt_matches = sorted(search_root.rglob('*.pt'))
    if pt_matches:
        return pt_matches[0]

    onnx_matches = sorted(search_root.rglob('weights.onnx')) or sorted(search_root.rglob('*.onnx'))
    if onnx_matches:
        return onnx_matches[0]

    return None


def _link_or_copy_weights(found: Path, weights_path: Path) -> Path:
    """Return a stable weights path, linking only when extensions match."""
    found = found.resolve()
    weights_path = Path(weights_path)
    if found == weights_path.resolve():
        return weights_path

    if found.suffix.lower() != weights_path.suffix.lower():
        print(f'Using cached weights at: {found}')
        return found

    weights_path.parent.mkdir(parents=True, exist_ok=True)
    if weights_path.exists():
        return weights_path

    try:
        weights_path.symlink_to(found)
        print(f'Linked {weights_path} -> {found}')
    except OSError:
        import shutil

        shutil.copy2(found, weights_path)
        print(f'Copied {found} -> {weights_path}')

    return weights_path


def _download_ev_weights_via_inference(
    cache_root: Path,
    *,
    api_key: str,
    project: str,
    version: int,
) -> Path | None:
    """Download and cache model weights via Roboflow Inference SDK."""
    cache_root.mkdir(parents=True, exist_ok=True)
    os.environ['MODEL_CACHE_DIR'] = str(cache_root)
    model_id = f'{project}/{version}'

    print(f'Downloading via Inference SDK (MODEL_CACHE_DIR={cache_root}) ...')
    from inference import get_model  # pyright: ignore[reportMissingImports]

    get_model(model_id=model_id, api_key=api_key)
    return find_ev_weights(cache_root)


def _download_ev_weights_via_roboflow_sdk(
    weights_path: Path,
    *,
    api_key: str,
    workspace: str,
    project: str,
    version: int,
) -> Path | None:
    """Try Roboflow Python SDK model.download() for raw PyTorch weights."""
    from roboflow import Roboflow  # pyright: ignore[reportMissingImports]

    download_dir = weights_path.parent
    download_dir.mkdir(parents=True, exist_ok=True)
    print("Trying Roboflow SDK model.download(format='pt') ...")
    rf = Roboflow(api_key=api_key)
    rf_model = rf.workspace(workspace).project(project).version(version).model
    rf_model.download(format='pt', location=str(download_dir))
    return find_ev_weights(download_dir)


def ensure_ev_model_weights(
    weights_path: Path,
    *,
    api_key: str = '',
    workspace: str = ROBOFLOW_WORKSPACE,
    project: str = ROBOFLOW_PROJECT,
    version: int = ROBOFLOW_VERSION,
    download: bool = False,
) -> Path:
    """Resolve local EV YOLO weights; optionally download from Roboflow once."""
    weights_path = Path(weights_path)
    search_root = weights_path.parent.parent

    if weights_path.exists():
        return weights_path

    found = find_ev_weights(search_root)
    if found is not None:
        print(f'Found EV weights at: {found}')
        return _link_or_copy_weights(found, weights_path)

    if not download:
        raise FileNotFoundError(
            f'EV model weights not found at {weights_path}.\n'
            'Run the one-time download cell with download=True, or place exported YOLOv8 weights there manually.'
        )

    api_key = (api_key or os.environ.get('ROBOFLOW_API_KEY', '')).strip()
    if not api_key:
        raise ValueError(
            'ROBOFLOW_API_KEY is required for one-time download. '
            'Set it in your environment before running the download cell.'
        )

    EV_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    errors: list[str] = []

    try:
        found = _download_ev_weights_via_inference(
            EV_MODEL_DIR,
            api_key=api_key,
            project=project,
            version=version,
        )
        if found is not None:
            print(f'Downloaded weights via Inference SDK: {found}')
            return _link_or_copy_weights(found, weights_path)
    except Exception as exc:
        errors.append(f'Inference SDK: {exc}')

    try:
        found = _download_ev_weights_via_roboflow_sdk(
            weights_path,
            api_key=api_key,
            workspace=workspace,
            project=project,
            version=version,
        )
        if found is not None:
            print(f'Downloaded weights via Roboflow SDK: {found}')
            return _link_or_copy_weights(found, weights_path)
    except Exception as exc:
        errors.append(f'Roboflow SDK: {exc}')

    msg = (
        f'Could not download EV model weights under {EV_MODEL_DIR}.\n'
        'Tried Inference SDK cache and Roboflow model.download().\n'
    )
    if errors:
        msg += 'Errors:\n- ' + '\n- '.join(errors) + '\n'
    msg += f'Manual fallback: place exported YOLOv8 weights at {weights_path}.'
    raise FileNotFoundError(msg)


def _normalize_class_name(name: str) -> str:
    return str(name).strip().replace(' ', '_')


@dataclass
class LocalYOLOEVClassifier:
    """Fine-tuned EV detector loaded from a local YOLO checkpoint."""

    weights_path: str | Path = EV_MODEL_WEIGHTS
    non_ev_classes: set[str] = field(default_factory=lambda: set(NON_EV_CLASS_NAMES))
    confidence_threshold: float = EV_CONFIDENCE_THRESHOLD
    inference_confidence: float = EV_INFERENCE_CONFIDENCE
    model: YOLO | None = field(default=None, repr=False)

    def __post_init__(self) -> None:
        weights = Path(self.weights_path)
        if not weights.exists():
            weights = ensure_ev_model_weights(weights)
        self.weights_path = str(weights)
        self.non_ev_classes = {_normalize_class_name(c) for c in self.non_ev_classes}
        print(f'Loading local EV YOLO model: {self.weights_path} ...')
        self.model = YOLO(self.weights_path)
        names = self.model.names
        class_list = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
        print(f'EV model classes ({len(class_list)}): {class_list}')

    def _is_non_ev(self, class_name: str) -> bool:
        normalized = _normalize_class_name(class_name)
        if normalized in self.non_ev_classes:
            return True
        return 'non' in normalized.lower() and 'ev' in normalized.lower()

    def _class_name_for_id(self, cls_id: int) -> str:
        names = self.model.names if self.model is not None else {}
        if isinstance(names, dict):
            return str(names.get(cls_id, cls_id))
        return str(names[cls_id])

    def score_crop(self, crop_bgr: np.ndarray) -> tuple[float, str | None]:
        """Return (ev_confidence, ev_brand) for a BGR vehicle crop."""
        if self.model is None:
            raise RuntimeError('EV model is not loaded.')

        results = self.model.predict(crop_bgr, conf=self.inference_confidence, verbose=False)
        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return 0.0, None

        best_ev_conf = 0.0
        best_ev_class: str | None = None
        best_non_ev_conf = 0.0

        for box in result.boxes:
            cls_id = int(box.cls[0].item())
            class_name = self._class_name_for_id(cls_id)
            conf = float(box.conf[0].item())
            if self._is_non_ev(class_name):
                best_non_ev_conf = max(best_non_ev_conf, conf)
            elif conf > best_ev_conf:
                best_ev_conf = conf
                best_ev_class = class_name

        if best_ev_conf < self.confidence_threshold:
            return 0.0, None
        if best_non_ev_conf > best_ev_conf:
            return 0.0, None

        return best_ev_conf, best_ev_class


def _clip_bbox(x1: int, y1: int, x2: int, y2: int, frame_shape: tuple[int, int, int]) -> tuple[int, int, int, int] | None:
    h, w = frame_shape[:2]
    x1 = max(0, min(x1, w - 1))
    x2 = max(0, min(x2, w))
    y1 = max(0, min(y1, h - 1))
    y2 = max(0, min(y2, h))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def safe_crop(frame: np.ndarray, x1: int, y1: int, x2: int, y2: int) -> np.ndarray | None:
    """Return a valid tight BGR crop or None if dimensions are too small."""
    clipped = _clip_bbox(x1, y1, x2, y2, frame.shape)
    if clipped is None:
        return None
    x1, y1, x2, y2 = clipped
    crop = frame[y1:y2, x1:x2]
    if crop.shape[0] < EV_MIN_CROP_SIDE_PX or crop.shape[1] < EV_MIN_CROP_SIDE_PX:
        return None
    return crop


def padded_vehicle_crop(
    frame: np.ndarray,
    x1: int,
    y1: int,
    x2: int,
    y2: int,
    pad_ratio: float = EV_CROP_PAD_RATIO,
) -> np.ndarray | None:
    """Return a padded vehicle crop for stage-2 EV object detection."""
    tight = safe_crop(frame, x1, y1, x2, y2)
    if tight is None:
        return None

    pad = int(pad_ratio * max(x2 - x1, y2 - y1))
    clipped = _clip_bbox(x1 - pad, y1 - pad, x2 + pad, y2 + pad, frame.shape)
    if clipped is None:
        return None
    px1, py1, px2, py2 = clipped
    return frame[py1:py2, px1:px2]


def update_ev_classification(
    track_ev_state: dict[int, dict],
    track_id: int,
    frame_idx: int,
    crop_bgr: np.ndarray,
    ev_classifier: LocalYOLOEVClassifier,
) -> dict:
    """Run fine-tuned EV model on a vehicle crop and update per-track voting state."""
    state = track_ev_state.get(track_id)
    if state is None:
        state = {
            'is_ev': None,
            'ev_votes': 0,
            'last_classify_frame': -9999,
            'last_ev_score': 0.0,
            'last_ev_brand': None,
            'score_ema': 0.0,
        }

    if state['is_ev'] is True:
        track_ev_state[track_id] = state
        return state

    should_classify = (frame_idx - state['last_classify_frame']) >= EV_CLASSIFY_EVERY_N_FRAMES
    if should_classify:
        ev_score, ev_brand = ev_classifier.score_crop(crop_bgr)
        state['last_ev_score'] = ev_score
        state['last_ev_brand'] = ev_brand
        state['last_classify_frame'] = frame_idx
        alpha = 0.35
        state['score_ema'] = ev_score if state['score_ema'] == 0.0 else (alpha * ev_score + (1 - alpha) * state['score_ema'])

        if ev_score >= EV_CONFIDENCE_THRESHOLD:
            state['ev_votes'] += 1
        else:
            state['ev_votes'] = max(0, state['ev_votes'] - 1)

        if state['ev_votes'] >= EV_MIN_VOTES and state['score_ema'] >= EV_CONFIDENCE_THRESHOLD:
            state['is_ev'] = True

    track_ev_state[track_id] = state
    return state

## 4b) Optional One-Time EV Model Download

Run this only if `EV_MODEL_WEIGHTS` does not already exist. It needs `ROBOFLOW_API_KEY` in your environment or assigned in the configuration cell.

In [ ]:
# resolved_ev_weights = ensure_ev_model_weights(
#     EV_MODEL_WEIGHTS,
#     api_key=ROBOFLOW_API_KEY,
#     workspace=ROBOFLOW_WORKSPACE,
#     project=ROBOFLOW_PROJECT,
#     version=ROBOFLOW_VERSION,
#     download=True,
# )
# resolved_ev_weights

## 5) Run Consolidated Processing

This cell writes the consolidated annotated video and CSV. Each interval counts each tracked object once per class, keeps the nearest observed proximity in that interval, and marks the interval entry stopped if stopped status is observed at least once.

In [ ]:
video_path = Path(VIDEO_PATH)
if not video_path.exists():
    raise FileNotFoundError(f'Video file not found: {video_path.resolve()}')

_start_ts = pd.to_datetime(START_DATETIME, utc=True, errors='coerce')
if pd.isna(_start_ts):
    raise ValueError(
        "Invalid START_DATETIME. Use ISO 8601 with UTC offset, e.g. '2026-01-19 07:50:22+00:00' or '2026-01-19T07:50:22Z'."
    )
start_datetime = _start_ts.to_pydatetime()

if CLOSE_AREA_THRESHOLD_PX2 < MEDIUM_AREA_THRESHOLD_PX2:
    raise ValueError('CLOSE_AREA_THRESHOLD_PX2 must be >= MEDIUM_AREA_THRESHOLD_PX2')

print('Loading YOLO model (stage 1: vehicle detection + tracking)...')
vehicle_model = YOLO(MODEL_WEIGHTS)
model_names = vehicle_model.names

print('Loading local EV YOLO model (stage 2: fine-tuned EV vs Non_EV detector)...')
ev_weights_path = globals().get('resolved_ev_weights', EV_MODEL_WEIGHTS)
ev_classifier = LocalYOLOEVClassifier(
    weights_path=ev_weights_path,
    confidence_threshold=EV_CONFIDENCE_THRESHOLD,
    inference_confidence=EV_INFERENCE_CONFIDENCE,
)

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError('Could not open video.')

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_seconds = total_frames / fps if fps > 0 else 0

print(f'FPS: {fps:.2f}')
print(f'Resolution: {frame_width}x{frame_height}')
print(f'Total frames: {total_frames}')
print(f'Duration (s): {duration_seconds:.2f}')
print(f'Start datetime: {format_wall_clock(start_datetime)}')
print(f'CSV columns: {CSV_COLUMNS}')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, fps, (frame_width, frame_height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f'Could not open video writer for: {OUTPUT_VIDEO_PATH}')

rows: list[dict[str, int | str]] = []
current_interval_index = 0
interval_unique_tracks: dict[int, dict[tuple[str, int], dict]] = {}
track_motion_state: dict[tuple[str, int], dict] = {}
track_ev_state: dict[int, dict] = {}
prev_gray: np.ndarray | None = None


def flush_interval(interval_idx: int) -> None:
    """Advance interval bookkeeping; final CSV rows are built after all EV promotions."""
    return None


frame_idx = 0
video_pbar = tqdm(
    total=total_frames if total_frames > 0 else None,
    desc='Processing consolidated video',
    unit='frame',
    dynamic_ncols=True,
)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        raw_frame = frame.copy()
        timestamp_seconds = int(frame_idx / fps) if fps > 0 else 0
        timestamp_float = frame_idx / fps if fps > 0 else 0.0
        current_datetime = start_datetime + timedelta(seconds=timestamp_float)
        interval_index = timestamp_seconds // SAMPLE_EVERY_SECONDS

        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        scene_motion_px = estimate_scene_motion_px(prev_gray, curr_gray) if prev_gray is not None else float('inf')
        scene_is_static = scene_motion_px <= SCENE_STATIC_MAX_MOTION_PX
        prev_gray = curr_gray

        while interval_index > current_interval_index:
            flush_interval(current_interval_index)
            current_interval_index += 1

        if interval_index not in interval_unique_tracks:
            interval_unique_tracks[interval_index] = {}
        interval_records = interval_unique_tracks[interval_index]

        results = vehicle_model.track(
            frame,
            conf=CONFIDENCE_THRESHOLD,
            persist=True,
            tracker=TRACKER_CONFIG,
            verbose=False,
        )
        result = results[0]

        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                if box.id is None:
                    continue

                track_id = int(box.id[0].item())
                cls_id = int(box.cls[0].item())
                cls_name = model_names.get(cls_id, str(cls_id)) if isinstance(model_names, dict) else model_names[cls_id]
                if cls_name not in VEHICLE_CLASSES:
                    continue

                conf = float(box.conf[0].item())
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

                width = max(0, x2 - x1)
                height = max(0, y2 - y1)
                bbox_area = width * height
                proximity = classify_proximity(
                    bbox_area,
                    close_threshold=CLOSE_AREA_THRESHOLD_PX2,
                    medium_threshold=MEDIUM_AREA_THRESHOLD_PX2,
                )
                center = ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

                gas_class_name = GAS_CLASS_NAMES[cls_name]
                stage1_track_key = (gas_class_name, track_id)

                crop = padded_vehicle_crop(raw_frame, x1, y1, x2, y2)
                if crop is not None:
                    ev_state = update_ev_classification(track_ev_state, track_id, frame_idx, crop, ev_classifier)
                else:
                    ev_state = track_ev_state.get(
                        track_id,
                        {
                            'is_ev': None,
                            'last_ev_score': 0.0,
                            'last_ev_brand': None,
                            'score_ema': 0.0,
                        },
                    )

                is_ev = ev_state.get('is_ev') is True
                output_class_name = EV_CLASS_NAME if is_ev else gas_class_name
                output_track_key = (output_class_name, track_id)

                if is_ev:
                    promote_track_records_to_ev(
                        interval_unique_tracks,
                        gas_class_name=gas_class_name,
                        track_id=track_id,
                    )
                    if output_track_key not in track_motion_state and stage1_track_key in track_motion_state:
                        track_motion_state[output_track_key] = track_motion_state.pop(stage1_track_key)
                    else:
                        track_motion_state.pop(stage1_track_key, None)

                motion_state = update_motion_state(
                    track_motion_state,
                    output_track_key,
                    center=center,
                    bbox_area=bbox_area,
                    timestamp_float=timestamp_float,
                    scene_is_static=scene_is_static,
                )
                update_interval_record(
                    interval_records,
                    output_track_key,
                    class_name=output_class_name,
                    proximity=proximity,
                    stopped=motion_state['stopped'],
                )

                color = EV_OVERLAY_COLOR if is_ev else PROXIMITY_COLORS[proximity]
                status_label = 'stopped' if motion_state['stopped'] else 'moving'
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3 if is_ev else 2)
                if is_ev:
                    label = f'ev | {proximity} | {status_label}'
                else:
                    label = (
                        f'{output_class_name} #{track_id} | {proximity} | {status_label} | '
                        f'scene_motion={scene_motion_px:.2f}px | conf={conf:.2f}'
                    )
                cv2.putText(
                    frame,
                    label,
                    (x1, max(20, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2,
                    cv2.LINE_AA,
                )

        confirmed_ev_tracks = sum(1 for state in track_ev_state.values() if state.get('is_ev') is True)
        overlay_text = (
            f'time={format_wall_clock(current_datetime)} | interval={current_interval_index} | '
            f"scene={'static' if scene_is_static else 'moving'} ({scene_motion_px:.2f}px) | "
            f'confirmed_ev_tracks={confirmed_ev_tracks}'
        )
        cv2.putText(
            frame,
            overlay_text,
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        writer.write(frame)
        frame_idx += 1
        if fps > 0:
            video_pbar.set_postfix(
                video=f'{timestamp_float:.1f}s / {duration_seconds:.1f}s',
                ev=confirmed_ev_tracks,
                refresh=False,
            )
        video_pbar.update(1)
finally:
    video_pbar.close()
    cap.release()
    writer.release()

if frame_idx > 0:
    final_timestamp_seconds = int((frame_idx - 1) / fps) if fps > 0 else 0
    final_interval_index = final_timestamp_seconds // SAMPLE_EVERY_SECONDS
    while current_interval_index <= final_interval_index:
        flush_interval(current_interval_index)
        current_interval_index += 1

    rows = []
    for interval_idx in range(final_interval_index + 1):
        interval_start_seconds = interval_idx * SAMPLE_EVERY_SECONDS
        interval_datetime = start_datetime + timedelta(seconds=interval_start_seconds)
        unique_tracks = interval_unique_tracks.get(interval_idx, {})
        counts = build_interval_counts_from_tracks(unique_tracks)
        rows.append(interval_row_template(format_wall_clock(interval_datetime), counts))
else:
    rows = []

df = pd.DataFrame(rows, columns=CSV_COLUMNS)
df.to_csv(str(OUTPUT_CSV_PATH), index=False)

print('Processing complete.')
print(f'Annotated video saved to: {Path(OUTPUT_VIDEO_PATH).resolve()}')
print(f'CSV saved to: {Path(OUTPUT_CSV_PATH).resolve()}')
print(f'Rows written: {len(df)}')
print(f'Confirmed EV tracks: {sum(1 for state in track_ev_state.values() if state.get("is_ev") is True)}')

## 6) Inspect Results

In [ ]:
if Path(OUTPUT_CSV_PATH).exists():
    result_df = pd.read_csv(OUTPUT_CSV_PATH)
    display(result_df.head())
    display(result_df.tail())
    print(result_df.columns.tolist())
else:
    print('CSV has not been created yet. Run the processing cell first.')

## 7) Tuning Guidance

- Increase `CONFIDENCE_THRESHOLD` to reduce false vehicle detections; decrease it to keep more Stage 1 vehicles before they are named `gas_car`, `gas_bus`, or `gas_truck`.
- Increase `EV_CONFIDENCE_THRESHOLD`, `EV_MIN_VOTES`, or `EV_CLASSIFY_EVERY_N_FRAMES` to make EV confirmation more conservative.
- Lower `EV_CONFIDENCE_THRESHOLD` or `EV_MIN_VOTES` if the EV detector misses known EVs. Once confirmed, a track is promoted to `ev` and no longer counted as a gas vehicle.
- Tune `CLOSE_AREA_THRESHOLD_PX2` and `MEDIUM_AREA_THRESHOLD_PX2` for your camera. The CSV uses `close` for the source notebooks' near/close bucket.
- Stopped status requires both a stable vehicle box and a static surrounding scene for at least `STOPPED_SECONDS`.